# Model Comparison

This notebook trains and evaluates all six forecasting models from the tutorial series on the same EEG dataset and train/test split, enabling a direct comparison of their forecast accuracy. The deep learning models (TCN, GRU, Transformer, S6/Mamba) are trained on a shared training set and evaluated on a held-out test set. The statistical models (DMD and VAR) are fit independently on each test trial's context window, since they operate on a per-trial basis. All model classes are imported from `utils/models.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from statsmodels.tsa.api import VAR
import scipy.linalg as sp_linalg
import math
from tqdm import trange, tqdm
import matplotlib.pyplot as plt

# import all custom model classes from the shared utils module
from utils.models import DMD, TCN, GRUForecaster, TimeSeriesTransformer, MambaForecaster

## Data Loading and Shared Split

We use a fixed random seed so every model trains and is evaluated on exactly the same trials.

In [ ]:
# load single subject from Gu et al., 2024 SSVEP experiment (downsampled to 250 Hz)
data: np.ndarray = np.load('../dataset/Data/data_s1_64_down.npy')

# dimensions: block x stimulation frequency x time x channels x conditions
nBlocks, nFreqs, nTime, nChans, nCons = data.shape

# select high luminance condition and collapse block/freq axes into a single trial axis
con_idx: int = 1
X: np.ndarray = data[:, :, :, :, con_idx].reshape(-1, nTime, nChans)
X = np.swapaxes(X, 1, 2)   # (trials, channels, time)

ssvep_chan_names = ['Pz', 'PO5', 'PO3', 'POz', 'PO4', 'PO6', 'O1', 'Oz', 'O2']
ssvep_chans_idx  = [48, 54, 55, 56, 57, 58, 61, 62, 63]
ssvep_chans_dict: dict = dict(zip(ssvep_chan_names, ssvep_chans_idx))

Fs: float = 250.                              # sampling frequency (Hz)
t: np.ndarray = np.arange(0, 5.14, 1 / Fs) * 1000   # time axis in ms

In [ ]:
# fixed seed ensures identical splits across all six models
np.random.seed(42)
nTrials: int = X.shape[0]
nTestTrials: int = math.ceil(nTrials * 0.2)
shuffIdx: np.ndarray = np.random.permutation(nTrials)

X_test_raw:  np.ndarray = X[shuffIdx[-nTestTrials:]]
X_train_val: np.ndarray = X[shuffIdx[:-nTestTrials]]

nValidTrials: int = math.ceil(X_train_val.shape[0] * 0.2)
X_val_raw:   np.ndarray = X_train_val[-nValidTrials:]
X_train_raw: np.ndarray = X_train_val[:-nValidTrials]

# normalise using training set statistics only to prevent data leakage
X_train_mu: float = X_train_raw.mean()
X_train_sd: float = X_train_raw.std()

X_train = torch.tensor((X_train_raw - X_train_mu) / X_train_sd, dtype=torch.float32)
X_val   = torch.tensor((X_val_raw   - X_train_mu) / X_train_sd, dtype=torch.float32)
X_test  = torch.tensor((X_test_raw  - X_train_mu) / X_train_sd, dtype=torch.float32)

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

In [ ]:
HORIZON_MS:  int = 1000
HORIZON:     int = int(abs(t - HORIZON_MS).argmin()) + 1   # number of samples in the forecast window
BATCH_SIZE:  int = 64
N_EPOCHS:    int = 150
UPDATE_FREQ: int = 50
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# minimal Dataset wrapper; returns full trial tensors as-is
class customDataSet(Dataset):
    def __init__(self, data: torch.Tensor):
        self.data = data
    def __len__(self) -> int:
        return len(self.data)
    def __getitem__(self, idx: int) -> torch.Tensor:
        return self.data[idx]

trainDL = DataLoader(customDataSet(X_train), batch_size=BATCH_SIZE, shuffle=True)
valDL   = DataLoader(customDataSet(X_val),   batch_size=BATCH_SIZE, shuffle=False)

## Shared Training Utilities

A single `train_loop` function handles all four deep learning models. The `channels_last` flag transposes batches to `(B, T, C)` before the forward pass for models that expect that layout (GRU, Transformer, S6/Mamba); TCN operates natively on `(B, C, T)` so the flag is `False` for that model.

In [ ]:
def train_loop(model: nn.Module, trainDL, valDL, n_epochs: int,
               update_freq: int, horizon: int, device,
               channels_last: bool = False):
    """Train for n_epochs; print val loss every update_freq epochs."""
    optimizer = optim.NAdam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    train_losses, val_losses = [], []

    for iEpoch in trange(n_epochs):
        model.train()
        epoch_loss: float = 0.0

        for batch in trainDL:
            # split trial into context (input) and forecast window (target)
            x = batch[:, :, :-horizon].to(device)
            y = batch[:, :, -horizon:].to(device)

            # transpose to (B, T, C) if the model expects channels last, then undo after
            pred = (model(x.transpose(1, 2)).transpose(1, 2)
                    if channels_last else model(x))

            loss = criterion(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(trainDL))

        if (iEpoch + 1) % update_freq == 0:
            model.eval()
            ev: float = 0.0
            with torch.no_grad():
                for batch in valDL:
                    x = batch[:, :, :-horizon].to(device)
                    y = batch[:, :, -horizon:].to(device)
                    pred = (model(x.transpose(1, 2)).transpose(1, 2)
                            if channels_last else model(x))
                    ev += criterion(pred, y).item()
            val_losses.append(ev / len(valDL))
            print(f'[{iEpoch + 1}/{n_epochs}]  train={train_losses[-1]:.3e}  val={val_losses[-1]:.3e}')

    return train_losses, val_losses


def get_dl_preds(model: nn.Module, X_test: torch.Tensor, horizon: int,
                 device, X_train_mu: float, X_train_sd: float,
                 channels_last: bool = False) -> np.ndarray:
    """Run inference on the test set and return predictions in µV."""
    model.eval()
    with torch.no_grad():
        x = X_test[:, :, :-horizon].to(device)
        preds = (model(x.transpose(1, 2)).transpose(1, 2)
                 if channels_last else model(x))
    # undo z-score normalisation to restore µV scale
    return ((preds * X_train_sd) + X_train_mu).cpu().numpy()

## Deep Learning Models

### TCN

The Temporal Convolutional Network uses dilated causal convolutions arranged in residual blocks. It takes input in `(B, C, T)` format, so `channels_last=False`.

In [ ]:
tcn_model = TCN(
    input_size=(nChans, nTime - HORIZON),
    horizon=HORIZON,
    nFilters=256,
    kernel_size=125,
    dropout=0.1,
).to(device)

tcn_train_loss, tcn_val_loss = train_loop(
    tcn_model, trainDL, valDL, N_EPOCHS, UPDATE_FREQ, HORIZON, device,
    channels_last=False
)
tcn_preds: np.ndarray = get_dl_preds(
    tcn_model, X_test, HORIZON, device, X_train_mu, X_train_sd, channels_last=False
)

### GRU

The GRU model uses stacked gated recurrent units followed by a direct multi-step linear head. It expects `(B, T, C)` input, so `channels_last=True`.

In [ ]:
gru_model = GRUForecaster(
    n_channels=nChans,
    hidden_size=256,
    num_layers=4,
    pred_len=HORIZON,
    dropout=0.05,
).to(device)

gru_train_loss, gru_val_loss = train_loop(
    gru_model, trainDL, valDL, N_EPOCHS, UPDATE_FREQ, HORIZON, device,
    channels_last=True
)
gru_preds: np.ndarray = get_dl_preds(
    gru_model, X_test, HORIZON, device, X_train_mu, X_train_sd, channels_last=True
)

### Transformer

The Informer-style Transformer uses ProbSparse attention and a distilling encoder stack to handle long EEG sequences efficiently. It expects `(B, T, C)` input, so `channels_last=True`.

In [ ]:
transformer_model = TimeSeriesTransformer(
    n_channels=nChans,
    d_model=128,
    n_heads=8,
    n_enc_layers=3,
    d_ff=256,
    pred_len=HORIZON,
    dropout=0.1,
).to(device)

transformer_train_loss, transformer_val_loss = train_loop(
    transformer_model, trainDL, valDL, N_EPOCHS, UPDATE_FREQ, HORIZON, device,
    channels_last=True
)
transformer_preds: np.ndarray = get_dl_preds(
    transformer_model, X_test, HORIZON, device, X_train_mu, X_train_sd, channels_last=True
)

### S6 / Mamba

The Mamba model uses selective state space layers with HiPPO initialisation. It expects `(B, T, C)` input, so `channels_last=True`.

In [ ]:
mamba_model = MambaForecaster(
    n_channels=nChans,
    d_model=64,
    d_state=16,
    n_layers=4,
    expand=2,
    pred_len=HORIZON,
    dropout=0.05,
).to(device)

mamba_train_loss, mamba_val_loss = train_loop(
    mamba_model, trainDL, valDL, N_EPOCHS, UPDATE_FREQ, HORIZON, device,
    channels_last=True
)
mamba_preds: np.ndarray = get_dl_preds(
    mamba_model, X_test, HORIZON, device, X_train_mu, X_train_sd, channels_last=True
)

## Statistical Models

DMD and VAR are fit independently on each test trial's context window (all time points before the forecast horizon). This mirrors their natural use case: fit once on the observed segment, then forecast forward.

### DMD

For each trial we detrend the context window, z-score normalise, fit a DMD model with Hankel time-delay embedding, extend the Vandermonde reconstruction into the horizon, and undo normalisation.

In [ ]:
dmd_preds_list = []

for trial in tqdm(X_test_raw, desc='DMD'):
    X_ctx: np.ndarray = trial[:, :-HORIZON].astype(float)   # (nChans, T_ctx)
    nT: int = X_ctx.shape[1]

    # fit and remove a linear trend across time
    design: np.ndarray = np.vstack([np.ones(nT), np.arange(nT)])
    beta, _, _, _ = sp_linalg.lstsq(design.T, X_ctx.T)
    trend: np.ndarray = (design.T @ beta).T   # (nChans, nT)

    # z-score normalise the detrended context
    mu_ctx: np.ndarray = X_ctx.mean(axis=-1, keepdims=True)
    sd_ctx: np.ndarray = X_ctx.std(axis=-1, keepdims=True) + 1e-8
    X_norm: np.ndarray = (X_ctx - trend - mu_ctx) / sd_ctx

    try:
        dmd = DMD(dt=1 / Fs, time_delay=50, svd_threshold=0.99,
                  scale_modes=True, clip_lambda=True)
        dmd.fit(X_norm)
        # reconstruct context + horizon, then take the last HORIZON columns
        pred_norm: np.ndarray = dmd.predict(
            start=0, end=X_norm.shape[-1] + HORIZON
        )[:, -HORIZON:]
        # undo normalisation; use the last trend value as a DC offset
        pred: np.ndarray = pred_norm * sd_ctx + mu_ctx + trend[:, -1:]
    except Exception:
        pred = np.zeros((nChans, HORIZON))

    dmd_preds_list.append(pred)

dmd_preds: np.ndarray = np.stack(dmd_preds_list)   # (nTestTrials, nChans, HORIZON)

### VAR

For each trial we apply first-order differencing to achieve approximate stationarity, standardise and reduce to 10 PCA components to handle multicollinearity across 64 channels, fit a VAR model with BIC lag selection, forecast the differenced PCA series, then invert PCA, standardisation, and cumulative sum to recover voltage-scale predictions.

In [ ]:
var_preds_list = []

for trial in tqdm(X_test_raw, desc='VAR'):
    X_ctx: np.ndarray = trial[:, :-HORIZON].T   # (T_ctx, nChans)

    # first-order difference to remove slow non-stationarities
    X_diff: np.ndarray = np.diff(X_ctx, axis=0)   # (T_ctx - 1, nChans)

    scaler = StandardScaler()
    X_scaled: np.ndarray = scaler.fit_transform(X_diff)

    # reduce to 10 PCA components to regularise the 64-channel VAR system
    pca = PCA(n_components=10)
    X_pca: np.ndarray = pca.fit_transform(X_scaled)

    try:
        var_result = VAR(X_pca).fit(maxlags=6, ic='bic', trend='c')
        lag: int = var_result.k_ar
        forecast_pca: np.ndarray = var_result.forecast(X_pca[-lag:], steps=HORIZON)

        # invert PCA and standardisation to get differenced channel predictions
        forecast_diff: np.ndarray = scaler.inverse_transform(
            pca.inverse_transform(forecast_pca)
        )   # (HORIZON, nChans)

        # invert differencing: cumsum and add last known observation
        last_val: np.ndarray = X_ctx[-1]   # (nChans,)
        forecast: np.ndarray = np.cumsum(forecast_diff, axis=0) + last_val[np.newaxis, :]
        var_preds_list.append(forecast.T)  # store as (nChans, HORIZON)
    except Exception:
        var_preds_list.append(np.zeros((nChans, HORIZON)))

var_preds: np.ndarray = np.stack(var_preds_list)   # (nTestTrials, nChans, HORIZON)

## Results

We compute the root mean squared error (RMSE) across all test trials, channels, and time steps for each model, then visualise the results as a bar chart and a 2x3 grid of trial-averaged forecasts at the Oz electrode.

In [ ]:
# ground-truth horizon window in original µV scale
X_test_truth: np.ndarray = X_test_raw[:, :, -HORIZON:]   # (nTestTrials, nChans, HORIZON)

def rmse(preds: np.ndarray, truth: np.ndarray) -> float:
    return float(np.sqrt(((preds - truth) ** 2).mean()))

model_names  = ['TCN', 'GRU', 'Transformer', 'S6 / Mamba', 'DMD', 'VAR']
model_preds  = [tcn_preds, gru_preds, transformer_preds, mamba_preds, dmd_preds, var_preds]
rmse_scores  = [rmse(p, X_test_truth) for p in model_preds]

print('Model Comparison — Test RMSE (µV)')
print('-' * 36)
for name, score in zip(model_names, rmse_scores):
    print(f'{name:>15}: {score:.3f}')

In [ ]:
# RMSE bar chart comparing all six models
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']

fig, ax = plt.subplots(figsize=(8, 4), dpi=100)
bars = ax.bar(model_names, rmse_scores, color=colors, edgecolor='k', linewidth=0.7)
ax.bar_label(bars, fmt='{:.2f}', padding=3, fontsize=9)
ax.set_ylabel(r'RMSE ($\mu V$)')
ax.set_title(f'Forecast RMSE by Model  (Horizon = {HORIZON_MS} ms)')
ax.set_ylim(0, max(rmse_scores) * 1.25)
plt.tight_layout()
plt.show()

In [ ]:
# trial-averaged forecast at Oz for each model in a 2x3 grid
ch: int = ssvep_chans_dict['Oz']
time_buffer: int = 100   # context samples shown before the horizon boundary

fig, axs = plt.subplots(2, 3, figsize=(14, 7), dpi=100, sharex=True, sharey=True)
axs_flat = axs.ravel()

# average context + horizon window from ground truth across test trials
truth_ctx_avg: np.ndarray = X_test_raw[:, ch, -(HORIZON + time_buffer):].mean(axis=0)

for i, (name, preds, color) in enumerate(zip(model_names, model_preds, colors)):
    avg_pred: np.ndarray = preds[:, ch, :].mean(axis=0)
    sem_pred: np.ndarray = preds[:, ch, :].std(axis=0) / np.sqrt(preds.shape[0])

    # plot context window from ground truth
    axs_flat[i].plot(t[-(HORIZON + time_buffer):], truth_ctx_avg,
                     color='k', alpha=0.4, label='Truth')
    # plot model forecast with ±1 SEM shading
    axs_flat[i].plot(t[-HORIZON:], avg_pred, color=color, alpha=0.85, label=name)
    axs_flat[i].fill_between(
        t[-HORIZON:], avg_pred - sem_pred, avg_pred + sem_pred,
        color=color, alpha=0.2, label=r'$\pm 1$ SEM'
    )
    # vertical dashed line marks the start of the forecast horizon
    axs_flat[i].axvline(t[-HORIZON], color='black', linestyle=':', linewidth=0.8)
    axs_flat[i].set_title(f'{name}  (RMSE={rmse_scores[i]:.2f})')

    if i >= 3:
        axs_flat[i].set_xlabel('Time (ms)')
    if i % 3 == 0:
        axs_flat[i].set_ylabel(r'Voltage ($\mu V$)')
    if i == 0:
        axs_flat[i].legend(frameon=False, fontsize=8)

plt.suptitle(f'Trial-Averaged Forecast Comparison — Oz  (Horizon = {HORIZON_MS} ms)', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

This notebook benchmarked all six forecasting methods from the tutorial series on identical data splits. The deep learning models (TCN, GRU, Transformer, S6/Mamba) were trained end-to-end to minimise MSE across many trials, giving them the advantage of learned representations but requiring substantial data. The statistical models (DMD, VAR) were fit per trial without any training data, relying instead on the linear dynamics present in the context window. Comparing RMSE across all models reveals how each algorithm's inductive biases align with the oscillatory, non-stationary structure of EEG: spectral methods like DMD can capture dominant frequency content without training data, while data-driven models that learn long-range temporal dependencies (GRU, S6/Mamba, Transformer) tend to generalise better when enough trials are available.